# Utilities - loss, training loop, timing

*Walkthrough notebook for `neural_operators/utils.py`.*

Every model notebook uses the same three helpers so that the comparison between FNO, GNO, LNO and MGNO is fair:

- **`RelativeL2Loss`** - the error metric the paper reports,
  $\|\text{pred}-\text{target}\|_2/\|\text{target}\|_2$ per sample, averaged over the batch.
- **`train_model`** - an Adam + cosine-annealing loop that records train/test loss each epoch.
- **`evaluate` / `measure_inference_time`** - final test error and average forward-pass time.

## Setup

The first cell makes the repository importable when this notebook is opened from `notebooks/source/`; the second holds the module's imports.

In [ ]:
# Make the repo root importable so `import neural_operators` works from notebooks/source/.
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "neural_operators").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

In [ ]:
import time
import torch

## The loss

Dividing by the norm of the target makes the error *relative*, so samples with large and small amplitudes
count equally. `eps` avoids division by zero.

In [ ]:
class RelativeL2Loss:
    """Per-sample relative L2 error, ||pred - target||_2 / ||target||_2, averaged over
    the batch. This is the discrete approximation to the Bochner norm error in Eq. (3)
    of the paper, and is the standard metric reported in Section 7."""

    def __init__(self, eps=1e-8):
        self.eps = eps

    def __call__(self, pred, target):
        pred = pred.reshape(pred.shape[0], -1)
        target = target.reshape(target.shape[0], -1)
        num = torch.norm(pred - target, p=2, dim=1)
        den = torch.norm(target, p=2, dim=1) + self.eps
        return (num / den).mean()

## Counting parameters

Used in the comparison table of notebook 06.

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## The training loop

Per epoch: loop over training batches (forward, loss, backward, optimizer step), step the learning-rate
schedule, then evaluate on the test set with gradients disabled. Returns the loss history for plotting.

In [ ]:
def train_model(
    model,
    train_loader,
    test_loader,
    epochs=100,
    lr=1e-3,
    weight_decay=1e-4,
    device="cpu",
    verbose_every=10,
):
    """Generic Adam + cosine-annealing training loop with relative-L2 loss, shared by
    every neural operator model (FNO, GNO, LNO, MGNO) so results are comparable."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    loss_fn = RelativeL2Loss()

    history = {"train_loss": [], "test_loss": []}
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        n_batches = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            n_batches += 1
        scheduler.step()
        train_loss /= n_batches

        model.eval()
        test_loss = 0.0
        n_test_batches = 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                test_loss += loss_fn(pred, y).item()
                n_test_batches += 1
        test_loss /= n_test_batches

        history["train_loss"].append(train_loss)
        history["test_loss"].append(test_loss)

        if verbose_every and (epoch % verbose_every == 0 or epoch == epochs - 1):
            elapsed = time.time() - t0
            print(
                f"epoch {epoch:4d} | train rel-L2 {train_loss:.4f} | "
                f"test rel-L2 {test_loss:.4f} | {elapsed:.1f}s"
            )

    return history

## Evaluation

`@torch.no_grad()` turns off gradient tracking, which saves memory and time at test time.

In [ ]:
def evaluate(model, loader, device="cpu"):
    model.eval()
    loss_fn = RelativeL2Loss()
    total = 0.0
    n_batches = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        total += loss_fn(pred, y).item()
        n_batches += 1
    return total / n_batches

## Timing

Runs a few warm-up passes first (the first calls are slower), synchronises the GPU if one is used, then
averages `n_repeats` forward passes.

In [ ]:
def measure_inference_time(model, x, device="cpu", n_repeats=20):
    model = model.to(device).eval()
    x = x.to(device)
    # warmup
    for _ in range(3):
        model(x)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_repeats):
        model(x)
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) / n_repeats

## Try it

Train a tiny linear model on a toy regression problem to see all the pieces work together.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)
x = torch.randn(256, 8)
y = x @ torch.randn(8, 1) + 3.0                       # a linear target (offset keeps ||y|| away from 0)
train = DataLoader(TensorDataset(x[:200], y[:200]), batch_size=32, shuffle=True)
test = DataLoader(TensorDataset(x[200:], y[200:]), batch_size=32)

model = torch.nn.Linear(8, 1)
print("parameters:", count_params(model))
history = train_model(model, train, test, epochs=30, lr=1e-2, verbose_every=10)
print("final test rel-L2:", evaluate(model, test))
print("inference time per batch (s):", measure_inference_time(model, x[:32]))

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history["train_loss"], label="train"); plt.plot(history["test_loss"], label="test")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("relative L2"); plt.legend(); plt.show()